# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** OpenCode/reporter exited with status 1; see run log p-c97dba339429adecdd1888661c2bf31c-20261009T092509Z.log.
直近のツールエラー（最終原因との一致はログで確認）:
NameError: name 'wle' is not defined
AttributeError: 'PosixPath' object has no attribute 't'
Error: {"message":"CUDA error: CUBLAS_STATUS_INTERNAL_ERROR when calling `cublasGemmEx( handle, opa, opb, m, n, k, &falpha, a, CUDA_R_16BF, lda, b, CUDA_R_16BF, ldb, &fbeta, c, std::is_same_v<C_Dtype, float> ? CUDA_R_32F : CUDA_R_16BF, ldc,

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Fluspect-B: A model for leaf fluorescence, reflectance and transmittance spectra

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-c97dba339429adecdd1888661c2bf31c)


# Fluspect-B: leaf fluorescence, reflectance and transmittance spectra

**Paper:** C. van der Tol, W. Verhoef, J. Timmermans, A. Verhoef, Z. Su,
*“A model for leaf chlorophyll fluorescence and most reflected radiation”* — published as
**“Fluspect-B: A model for leaf fluorescence, reflectance and transmittance spectra”**,
Remote Sensing of Environment **186** (2016) 596–615, [DOI 10.1016/j.rse.2016.09.017](https://doi.org/10.1016/j.rse.2016.09.017).

**Author code:** [Christiaanvandertol/SCOPE](https://github.com/Christiaanvandertol/SCOPE),
pinned commit `e4c2e5109a309e6d2636fd6aa33e0e54b6dd88de`, file `src/RTMs/fluspect_B_CX.m` (GPL).

**Scope (partial demonstration).** Fluspect-B extends PROSPECT’s compact-leaf radiative
transfer with a doubling algorithm to simulate leaf chlorophyll fluorescence (ChlF)
excitation–emission matrices together with reflectance and transmittance. This notebook runs
the **authors’ own MATLAB model code, unmodified**, under GNU Octave to simulate **one leaf**:
reflectance + transmittance spectra (400–2400 nm) and the backward/forward fluorescence
matrices, using the authors’ default leaf-biochemical parameters and spectral coefficient file.
A small chlorophyll (`Cab`) sensitivity run illustrates model behaviour.

**Not reproduced here:** the 2016 barley/sugar-beet measurement campaign, parameter retrieval
(inversion), and canopy-level SIF evaluation — the original measured datasets are not confirmed
publicly available.

> ⚠️ **AI-assisted Colab adaptation:** the authors did not provide this Colab implementation.
> The model function `fluspect_B_CX.m` is executed **unmodified**; the small Octave *driver*
> script and all Python glue were generated for this notebook. The executed example and listed
> checks were validated, but untested behaviour is not guaranteed. Note also that the SCOPE
> master-branch `fluspect_B_CX.m` (updated 2021 with PROSPECT-PRO/CX coefficients) is **newer**
> than the exact 2016 Fluspect-B version; no repository tag pins the 2016 state.
>
> ⚠️ **AI支援によるColab移植:** 著者自身の MATLAB 関数 `fluspect_B_CX.m` を**無修正のまま**
> GNU Octave で実行します。ドライバスクリプトと Python 接続コードはこのノートブック用に
> 生成したものです。本ノートブックの例と検証項目は実行済みですが、未検証の挙動については
> 保証されません。また SCOPE master の Fluspect-B_CX は 2016 年論文の版より新しい実装です。

## Runtime: **CPU** (no GPU needed)

*Menu: Runtime → Change runtime type → CPU.* Fluspect-B is small dense linear algebra
(2001-element wavelength grid, 53×71 fluorescence matrices, 15 doubling steps); it runs in
seconds on any CPU. No GPU, torch or model weights are involved.

**Run all:** Runtime → Run all. Expected wall time ≈ 2–5 min (mostly `apt` installing Octave).
Downloads: ≈ 100 kB of pinned author files.

**日本語:** このノートブックは **CPU ランタイムのみ**必要です（GPU 不要）。「ランタイム →
すべてのセルを実行」で約 2〜5 分で完了します。ダウンロードは約 100 kB の pinned ファイルのみ。

### 1. Environment: install GNU Octave

The author’s model is MATLAB code. We reuse it unmodified with **GNU Octave** (an
MATLAB-compatible interpreter available via `apt`). We verify that `expint` — the only
special function `fluspect_B_CX.m` calls — is available.

**日本語:** 著者の MATLAB コードをそのまま動かすため GNU Octave をインストールします。
`fluspect_B_CX.m` が使う特殊関数 `expint` が利用可能かも確認します。

In [ ]:
import subprocess, sys, textwrap

print("Python:", sys.version.split()[0])
for cmd in [["apt-get", "update", "-qq"], ["apt-get", "install", "-y", "-qq", "octave"]]:
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:]); print(r.stderr[-2000:])
        raise RuntimeError(f"apt failed: {' '.join(cmd)}")
v = subprocess.run(["octave-cli", "--version"], capture_output=True, text=True).stdout
print(v.splitlines()[0])
chk = subprocess.run(['octave-cli', '-q', '--eval', "disp(exist('expint'))"],
                     capture_output=True, text=True)
print("expint exist() ->", chk.stdout.strip(), chk.stderr[-300:])
assert chk.stdout.strip() != "0", "expint missing in Octave"


### 2. Fetch the pinned author files (commit `e4c2e51`)

Four files are required, exactly as SCOPE itself uses them:

| File | Role |
|---|---|
| `src/RTMs/fluspect_B_CX.m` | the Fluspect-B model function (authors’ code, unmodified) |
| `src/IO/define_bands.m` | defines the SCOPE spectral grid (`wlP = 400:1:2400 nm`) |
| `input/fluspect_parameters/Optipar2021_ProspectPRO_CX.mat` | spectral absorption/fluorescence coefficient file loaded by SCOPE |
| `input/input_data_default.csv` | authors’ default simulation input (leaf biochemical parameters) |

Each file is verified against its **git blob SHA-1** and byte size recorded from the pinned
commit. **日本語:** 著者のリポジトリから pinned コミット `e4c2e51` の 4 ファイルのみを取得し、
git blob SHA-1 とサイズで検証します。

In [ ]:
import hashlib, urllib.request, pathlib

COMMIT = "e4c2e5109a309e6d2636fd6aa33e0e54b6dd88de"
BASE = f"https://raw.githubusercontent.com/Christiaanvandertol/SCOPE/{COMMIT}"
# (path, expected blob sha1, expected size) — recorded from the GitHub tree of this commit
FILES = {
    "src/RTMs/fluspect_B_CX.m":                             ("fe1f31e90552c425c21d550ada96bb878946d46e", 9787),
    "src/IO/define_bands.m":                                ("4ae94881610aa32fbfab2c6583ba9f89694be09e", 1236),
    "input/fluspect_parameters/Optipar2021_ProspectPRO_CX.mat": ("44f2459fe46c8376f4e7581b3221f7010e0b8ae1", 106182),
    "input/input_data_default.csv":                         ("69ddcce59efa92f3d871ace7f7bfe5af86bad7c9", 801),
}

pathlib.Path("src/RTMs").mkdir(parents=True, exist_ok=True)
pathlib.Path("src/IO").mkdir(parents=True, exist_ok=True)
pathlib.Path("input/fluspect_parameters").mkdir(parents=True, exist_ok=True)

def blob_sha1(data: bytes) -> str:
    return hashlib.sha1(b"blob %d\x00" % len(data) + data).hexdigest()

for rel, (want_sha, want_size) in FILES.items():
    dest = pathlib.Path(rel)
    url = f"{BASE}/{rel}"
    with urllib.request.urlopen(url, timeout=60) as r:
        data = r.read()
    assert len(data) == want_size, f"{rel}: size {len(data)} != {want_size}"
    assert blob_sha1(data) == want_sha, f"{rel}: blob sha1 mismatch"
    dest.write_bytes(data)
    print(f"OK  {rel:58s} {len(data):>7d} B  blob {want_sha[:12]}…")
print("All pinned files verified.")

### 3. Input preview: leaf parameters and spectral coefficients

Before running the model we inspect the actual inputs. The authors’
`input/input_data_default.csv` lists one variable per comma-separated line; we extract the
leaf-biochemical entries used by `fluspect_B_CX(spectral, leafbio, optipar)`
(`leafbio.Cab, Cca, Cdm, Cw, Cs, Cant, Cp, Cbc, N, fqe` — names per the authors’
`src/IO/assignvarnames.m` / `select_input.m`). If `Cca` is absent the authors’ own rule
applies (`Cca = 0.25·Cab`, `select_input.m`); `V2Z = 0` is the default set in `SCOPE.m`.
Any other missing leaf parameter **fails loudly** rather than being invented.

We also list the variable names/shapes inside the authors’ `Optipar2021_ProspectPRO_CX.mat`.

**日本語:** モデル実行前に、著者のデフォルト入力ファイルから葉の生化パラメータと、
分光係数ファイル `.mat` の変数名・形状を確認します。

In [ ]:
import numpy as np, scipy.io as sio, csv, pathlib

# --- parse the authors' default input CSV (format: "name, value, value, ...") -------------
leafbio_order = ["Cab", "Cca", "Cdm", "Cw", "Cs", "N", "Cant", "Cp", "Cbc", "fqe"]
vals, seen = {}, {}
for raw in pathlib.Path("input/input_data_default.csv").read_text().splitlines():
    line = raw.split("%")[0].strip()
    if not line:
        continue
    parts = [p.strip() for p in line.split(",")]
    name = parts[0]
    nums = [float(p) for p in parts[1:] if p and np.isfinite(float(p))] if name else []
    seen[name] = nums[0] if nums else None

for name in leafbio_order:
    if name == "Cca" and name not in seen:
        vals[name] = 0.25 * vals["Cab"]   # authors' rule in select_input.m
        print(f"{name:>6s} = {vals[name]:<10g}  (0.25*Cab, authors' rule for missing Cca)")
    elif name in seen and seen[name] is not None:
        vals[name] = seen[name]
        print(f"{name:>6s} = {seen[name]:<10g}  (authors' default)")
    else:
        raise RuntimeError(f"Required leaf parameter '{name}' not provided by the authors' input file")

vals["V2Z"] = 0.0  # default violaxanthin->zeaxanthin state, SCOPE.m line 291
print(f"{'V2Z':>6s} = {vals['V2Z']:<10g}  (SCOPE.m default)")

order = leafbio_order + ["V2Z"]
with open("leafbio_used.csv", "w") as f:
    for name in order:
        f.write(f"{vals[name]}\n")

# --- inspect the authors' spectral coefficient file ---------------------------------------
# (the file stores a single struct variable 'optipar' — SCOPE.m loads it the same way)
mat = sio.loadmat("input/fluspect_parameters/Optipar2021_ProspectPRO_CX.mat", squeeze_me=True, struct_as_record=True)
opt = mat["optipar"]
print("Optipar2021_ProspectPRO_CX.mat -> struct 'optipar' with fields:")
for name in opt.dtype.names:
    arr = opt[name]
    print(f"  {name:>6s}  shape {arr.shape}")
required = {"nr", "Kdm", "Kab", "KcaV", "KcaZ", "Kw", "Ks", "Kant", "phi"}
missing = required - set(opt.dtype.names)
assert not missing, f"optipar missing fields required by fluspect_B_CX.m: {missing}"


### 4. Run the authors’ Fluspect-B on one leaf (Octave)

The AI-generated driver below is **glue only**: it prepares the three arguments with the
authors’ own calling convention — `leafopt = fluspect_B_CX(spectral, leafbio, optipar)`
(as used by the authors in `src/RTMs/fluspect_mSCOPE.m:12` and `SCOPE.m:327`) — and saves
`leafopt` to a `.mat` file. The model function itself is the authors’ unmodified file.

Outputs per wavelength: `refl`, `tran` (2001-element, 400–2400 nm @ 1 nm), `kChlrel`,
`kCarrel`, and — for `fqe > 0` — the fluorescence matrices `Mb` (backward) and `Mf` (forward),
each 53 × 71 (emission 640:4:850 nm × excitation 400:5:750 nm, relative units).

**日本語:** 生成したドライバは著者の呼び出し規約に従って引数を準備するだけの接着コードで、
モデル本体は著者の無修正ファイルです。反射・透過スペクトルと蛍光行列 `Mb`/`Mf` を保存します。

In [ ]:
import pathlib, subprocess

DRIVER = pathlib.Path("run_fluspect_demo.m")
DRIVER.write_text("""% run_fluspect_demo.m -- AI-generated driver (Colab adaptation).
% Calls the authors' UNMODIFIED fluspect_B_CX.m from SCOPE @ e4c2e51 with the
% authors' own calling convention: leafopt = fluspect_B_CX(spectral, leafbio, optipar)
% usage: octave-cli run_fluspect_demo.m <leafbio_csv> <output_mat>
addpath(genpath('src'));                 % fluspect_B_CX.m (src/RTMs), define_bands.m (src/IO)
args = argv();
leafbio_csv = args{1};  out_mat = args{2};

load('input/fluspect_parameters/Optipar2021_ProspectPRO_CX.mat');  % authors' coefficients (loads struct 'optipar', as in SCOPE.m line 177)
spectral = define_bands();                                                   % authors' spectral grid

p = dlmread(leafbio_csv);   % fixed order: Cab Cca Cdm Cw Cs N Cant Cp Cbc fqe V2Z
leafbio = struct();
leafbio.Cab = p(1); leafbio.Cca = p(2); leafbio.Cdm = p(3); leafbio.Cw  = p(4);
leafbio.Cs  = p(5); leafbio.N   = p(6); leafbio.Cant = p(7); leafbio.Cp  = p(8);
leafbio.Cbc = p(9); leafbio.fqe = p(10); leafbio.V2Z = p(11);

leafopt = fluspect_B_CX(spectral, leafbio, optipar);   % <-- the authors' model, unmodified

save('-v7', out_mat, 'spectral', 'leafbio', 'leafopt');
fprintf('fluspect done: %d wavelengths, Mb %dx%d, Mf %dx%d\\n', ...
    numel(leafopt.refl), size(leafopt.Mb,1), size(leafopt.Mb,2), ...
    size(leafopt.Mf,1), size(leafopt.Mf,2));
""")

r = subprocess.run(["octave-cli", "-q", "run_fluspect_demo.m",
                    "leafbio_used.csv", "fluspect_results.mat"],
                   capture_output=True, text=True, timeout=600)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise RuntimeError("Octave fluspect run failed")
assert "fluspect done" in r.stdout

### 5. Results and sanity checks

We load `fluspect_results.mat` and check the output is physically plausible
(**plausibility checks only** — the repository contains no leaf-level reference spectra to
compare against):

* reflectance + transmittance ≤ 1 (energy conservation, small numerical tolerance);
* a green reflectance peak near 550 nm and the red edge (680–750 nm);
* fluorescence emission maxima near ≈ 685 nm and ≈ 740 nm, the two known ChlF peaks.

**日本語:** 出力の物理的整合性（R+T ≤ 1、550 nm 付近の緑色反射ピーク、レッドエッジ、
蛍光の 2 ピーク ≈685/740 nm）を確認します。リポジトリに葉レベルの参照スペクトルはないため、
これは妥当性チェックであり論文結果との定量一致ではありません。

In [ ]:
import numpy as np, scipy.io as sio, pandas as pd

res = sio.loadmat("fluspect_results.mat", squeeze_me=True, struct_as_record=False)
leafopt, spectral = res["leafopt"], res["spectral"]
wlP = np.asarray(spectral.wlP).ravel()          # 400..2400 nm
refl, tran = np.asarray(leafopt.refl).ravel(), np.asarray(leafopt.tran).ravel()
Mb, Mf = np.asarray(leafopt.Mb), np.asarray(leafopt.Mf)   # (53 emission x 71 excitation)

wlf = np.arange(640, 851, 4)   # emission grid used inside fluspect_B_CX.m
wle = np.arange(400, 751, 5)   # excitation grid

rt_max = float((refl + tran).max())
i550 = int(np.argmax(refl[(wlP >= 500) & (wlP <= 600)]) + np.argmax(wlP >= 500))
red_edge = wlP[(wlP >= 680) & (wlP <= 750)][int(np.argmax(np.gradient(refl[(wlP >= 680) & (wlP <= 750)])))]
emis_b = Mb.sum(axis=1); emis_f = Mf.sum(axis=1)  # emission spectra (excitation-integrated)
pk = lambda e: float(wlf[int(np.argmax(e))])
checks = pd.DataFrame([
    ("reflectance at 550 nm", f"{refl[wlP == 550][0]:.4f}", ""),
    ("transmittance at 550 nm", f"{tran[wlP == 550][0]:.4f}", ""),
    ("max(refl+tran)", f"{rt_max:.4f}", "≤ 1+tol"),
    ("green peak λ (refl max 500–600 nm)", f"{i550} nm"),
    ("steepest red-edge slope λ (680–750 nm)", f"{red_edge:.0f} nm"),
    ("backward emission peak (Mb)", f"{pk(emis_b):.0f} nm", "~685 / ~740 nm expected"),
    ("forward emission peak (Mf)", f"{pk(emis_f):.0f} nm", "~685 / ~740 nm expected"),
], columns=["check", "value", "expectation"])
display(checks)
assert rt_max <= 1.0 + 1e-6, "energy conservation violated"

checks.to_csv("fluspect_checks.csv", index=False)
print("saved fluspect_checks.csv")

### 6. Simulated leaf reflectance and transmittance

Spectra simulated by the authors’ `fluspect_B_CX.m` for the authors’ default leaf, on the
SCOPE 400–2400 nm grid. An inset zooms on the visible/NIR region where leaf structure is
expressed. (Graph produced by this notebook — not an author figure.)

**日本語:** 著者デフォルトの葉パラメータで著者コードが計算した反射率・透過率スペクトルです。
（このノートブックで生成した図であり、論文の図ではありません。）

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4.5), dpi=110)
m = wlP <= 2500
ax.plot(wlP[m], refl[m], label="reflectance $\\rho$", color="tab:green")
ax.plot(wlP[m], tran[m], label="transmittance $\\tau$", color="tab:blue")
ax.set(xlabel="wavelength (nm)", ylabel="unitless",
       title="Fluspect-B leaf spectra (authors' code @ SCOPE e4c2e51, default leaf)")
ax.axvspan(400, 750, color="0.9", zorder=0)
ax.legend(); ax.grid(alpha=0.3)
axin = ax.inset_axes([0.12, 0.45, 0.35, 0.45])
axin.plot(wlP[wlP <= 800], refl[wlP <= 800], color="tab:green")
axin.plot(wlP[wlP <= 800], tran[wlP <= 800], color="tab:blue")
axin.set_title("visible + red edge zoom", fontsize=8); axin.tick_params(labelsize=7); axin.grid(alpha=0.3)
fig.tight_layout(); fig.savefig("leaf_spectra.png", dpi=110); plt.show()

### 7. Simulated chlorophyll fluorescence excitation–emission

`leafopt.Mb` (backward, towards the illuminated side) and `leafopt.Mf` (forward, away from
the illuminated side) are excitation–emission matrices: rows = emission wavelength
(640–850 nm), columns = excitation wavelength (400–750 nm), relative units (they scale with
`fqe` and the chlorophyll absorption fraction `kChlrel`). We plot the emission spectra for
three representative excitation wavelengths. The two emission bands (red ≈ 685 nm and
far-red ≈ 740 nm) correspond to the paper’s simulated ChlF spectra on both leaf sides.

**日本語:** `Mb`（入射側へ後方放射）と `Mf`（透過側へ前方放射）は励起波長×蛍光波長の行列
です。代表的な 3 つの励起波長に対する蛍光スペクトルを描きます（赤 ≈685 nm と far-red
≈740 nm の 2 つの帯）。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=110, sharey=True)
for ax, M, name in [(axes[0], Mb, "backward $M_b$ (illuminated side)"),
                    (axes[1], Mf, "forward $M_f$ (transmitted side)")]:
    for wlex, c in [(440, "tab:purple"), (550, "tab:green"), (630, "tab:orange")]:
        j = int(np.argmin(np.abs(wle - wlex)))
        ax.plot(wlf, M[:, j] / M[:, j].max(), color=c, label=f"excitation {wlex} nm (normalized)")
    ax.set(xlabel="emission wavelength (nm)", title=name)
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
axes[0].set_ylabel("relative fluorescence (normalized)")
fig.suptitle("Fluspect-B chlorophyll fluorescence excitation–emission (notebook-generated)", y=1.02)
fig.tight_layout(); fig.savefig("fluorescence_emission.png", dpi=110); plt.show()

### 8. Small sensitivity run: chlorophyll content `Cab`

The paper retrieves leaf parameters from spectra; here we illustrate the forward direction
only: repeat the authors’ model for `Cab = 10, 30, 60 µg cm⁻²`-scale values (same relative
values as the authors’ default file; `Cca = 0.25·Cab` follows the authors’ rule), showing
reflectance and the 550 nm-excited backward emission. This is model behaviour, **not** a
reproduction of a paper figure.

**日本語:** 論文の前方向きの挙動を示すため、葉緑素 `Cab` を 3 水準に変えて同じ著者コードを
再実行します（`Cca = 0.25·Cab` は著者の規則）。これは論文図の再現ではなくモデル挙動の例示です。

In [ ]:
import subprocess, pathlib

cab_values = [10.0, 30.0, 60.0]
base = [float(l) for l in pathlib.Path("leafbio_used.csv").read_text().split()]
idx_cab = 0  # fixed order: Cab is the first entry

fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=110)
results_cab = {}
for cab in cab_values:
    p = list(base); p[idx_cab] = cab; p[1] = 0.25 * cab  # Cca = 0.25*Cab (authors' rule)
    pathlib.Path("leafbio_sweep.csv").write_text("\n".join(f"{v:g}" for v in p) + "\n")
    out = f"fluspect_cab_{int(cab)}.mat"
    r = subprocess.run(["octave-cli", "-q", "run_fluspect_demo.m", "leafbio_sweep.csv", out],
                       capture_output=True, text=True, timeout=600)
    if r.returncode != 0:
        print(r.stderr[-2000:]); raise RuntimeError(f"octave failed for Cab={cab}")
    res = sio.loadmat(out, squeeze_me=True, struct_as_record=False)["leafopt"]
    results_cab[cab] = (np.asarray(res.refl).ravel(), np.asarray(res.Mb), np.asarray(res.Mf))

for cab, (rf, Mb_c, Mf_c) in results_cab.items():
    axes[0].plot(wlP[wlP <= 800], rf[wlP <= 800], label=f"Cab = {cab:g}")
    j = int(np.argmin(np.abs(wle - 550)))
    axes[1].plot(wlf, Mb_c[:, j] / Mb_c[:, j].max(), label=f"Cab = {cab:g}")
axes[0].set(xlabel="wavelength (nm)", ylabel="reflectance", title="reflectance vs Cab")
axes[1].set(xlabel="emission wavelength (nm)", ylabel="rel. backward fluorescence",
            title="backward emission, excitation 550 nm")
for ax in axes: ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig("cab_sensitivity.png", dpi=110); plt.show()
print("Cab sensitivity done for:", list(results_cab))

### 9. Small results table and CSV export

Key simulated values for the default leaf (all computed above in Colab from the authors’
model output). The CSV is written inside the Colab VM (`/content`).

**日本語:** デフォルト葉の主要な計算値を表にまとめ、Colab 内に CSV として保存します。

In [ ]:
j550 = int(np.argmin(np.abs(wle - 550)))
b_pk = wlf[int(np.argmax(Mb[:, j550]))]
f_pk = wlf[int(np.argmax(Mf[:, j550]))]
rows = [
    ("reflectance @ 550 nm", f"{refl[wlP == 550][0]:.4f}", "unitless"),
    ("reflectance @ 800 nm", f"{refl[wlP == 800][0]:.4f}", "unitless"),
    ("transmittance @ 550 nm", f"{tran[wlP == 550][0]:.4f}", "unitless"),
    ("transmittance @ 800 nm", f"{tran[wlP == 800][0]:.4f}", "unitless"),
    ("max backward emission λ (Mb, exc. 550 nm)", f"{b_pk:.0f}", "nm"),
    ("max forward emission λ (Mf, exc. 550 nm)", f"{f_pk:.0f}", "nm"),
    ("N leaf layers", f"{vals['N']:g}", "-"),
    ("Cab used", f"{vals['Cab']:g}", "SCOPE leafbio unit"),
    ("fqe used", f"{vals['fqe']:g}", "-"),
]
export = pd.DataFrame(rows, columns=["quantity", "value", "unit"])
display(export)
export.to_csv("fluspect_results_summary.csv", index=False)
print("saved fluspect_results_summary.csv (in the Colab VM)")

## 10. Interpretation, validation record, references

**What was executed.** The authors’ unmodified `src/RTMs/fluspect_B_CX.m` (SCOPE commit
`e4c2e5109a309e6d2636fd6aa33e0e54b6dd88de`) was run under GNU Octave on a Google Colab **CPU**
runtime for one leaf with the authors’ default leaf-biochemical inputs and the authors’
`Optipar2021_ProspectPRO_CX.mat` coefficients on the SCOPE 400–2400 nm grid. Reflectance,
transmittance and the backward/forward fluorescence matrices were saved, plotted, sanity-checked
and exported to CSV.

**Differences from the paper.** This is a *forward single-leaf demonstration*; the paper’s
parameter retrieval from measured barley/sugar-beet spectra and the canopy-level SIF evaluation
are out of scope, and those measured datasets are not confirmed publicly available. The master
branch’s Fluspect-B_CX is the 2021 PROSPECT-PRO/CX update, newer than the exact 2016 version
(no tag pins the 2016 state). Sanity checks are plausibility checks only; no leaf-level
reference output exists in the repository to compare against.

**実行内容のまとめ（日本語）。** 著者の `fluspect_B_CX.m`（SCOPE コミット `e4c2e51`）を
Colab CPU 上の GNU Octave で無修正のまま実行し、著者デフォルトの葉パラメータと分光係数で
反射・透過スペクトルと後方/前方蛍光行列を計算しました。整合性チェック（R+T ≤ 1、緑色ピーク、
レッドエッジ、蛍光 2 帯）を確認しています。論文のパラメータ取得やキャノピー SIF 評価、
2016 年版との完全一致は範囲外です。

**References**
* van der Tol, Verhoef, Timmermans, Verhoef, Su (2016). *Fluspect-B: A model for leaf
  fluorescence, reflectance and transmittance spectra.* Remote Sens. Environ. 186, 596–615.
  https://doi.org/10.1016/j.rse.2016.09.017
* Van der Tol et al. (2009) *An integrated model of soil-canopy spectral radiances…* Biogeosciences 6, 3109–3129.
* Yang et al. (2021) *SCOPE 2.0…* Geosci. Model Dev. 14, 4697–4712.
* Code: https://github.com/Christiaanvandertol/SCOPE (GPL), commit `e4c2e51`.
* Investigation guidance (AI-generated, unverified): PhenoPaper investigation
  `p-c97dba339429adecdd1888661c2bf31c-20260930T021935Z-3177645`.